# Create Supplement eTables

This notebook creates publication-ready supplement tables:

- eTables 7-14: simulation-based power tables, formatted with rounded values and without an all-zero failed-fit column.
- eTables 16-19: covariate missingness by TTM group, formatted without `p_value` or `test` columns and with `*` marking significant differences.

Outputs are written to `summarized_results/supplement_etables/` as CSV files and a combined Excel workbook.

In [ ]:
import ast
import glob
import json
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, fisher_exact

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

In [ ]:
ROOT = Path.cwd()
if not (ROOT / "summarized_results").exists() and (ROOT.parent / "summarized_results").exists():
    ROOT = ROOT.parent

OUT = ROOT / "summarized_results" / "supplement_etables"
OUT.mkdir(parents=True, exist_ok=True)

POWER_INPUT_CANDIDATES = [
    ROOT / "summarized_results" / "power_simulation_outputs" / "power_simulation_raw.csv",
    ROOT / "pooled" / "power_simulation_outputs" / "power_simulation_raw.csv",
    ROOT / "power_simulation_outputs" / "power_simulation_raw.csv",
]

CSV = {
    "HYPERION": [ROOT / "hyperion" / "predictorsDf.csv"],
    "eICU": [ROOT / "eICU" / "eICUPredictorsDiag.csv", ROOT / "eICU" / "eICUPredictors.csv"],
    "PMAP": [ROOT / "pmap" / "PMAP_Predictors2.csv", ROOT / "pmap" / "PMAP_Predictors.csv"],
    "MIMIC-IV": [ROOT / "mimiciv" / "MIMIC_Predictors.csv", ROOT / "mimiciv" / "MIMIC_Predictors1.csv"],
}

DML_NOTEBOOK = {
    "HYPERION": ROOT / "hyperion" / "hyperionAnalysisDML.ipynb",
    "eICU": ROOT / "eICU" / "eICUAnalysisDML.ipynb",
    "PMAP": ROOT / "pmap" / "PMAPAnalysisDML.ipynb",
    "MIMIC-IV": ROOT / "mimiciv" / "MIMICAnalysisDML.ipynb",
}

TREATMENT_CANDIDATES = {
    "HYPERION": ["groupe", "TTM", "hypothermia"],
    "eICU": ["Hypothermia", "hypothermia", "treatment_hypothermia"],
    "PMAP": ["hypothermia", "TTM"],
    "MIMIC-IV": ["hypothermia", "TTM"],
}

OUTCOME_OR_ID_COLUMNS = {
    "patientunitstayid", "subject_id", "hadm_id", "stay_id", "osler_id", "pat_enc_csn_id", "SUBJID",
    "death_at_disch", "DeathAtDischarge", "hospital_mortality", "hospital_expire_flag",
    "LastMGCSPositive", "last_mGCS", "last_mGCS_time", "first_mGCS_time",
    "LastGCSPositive", "LastGCS15", "LastGCS", "LastMGCSTime", "FirstMGCSTime",
    "CPC_SC3", "CPC12", "BARTHEL_SC", "SOFA_SC7", "DS_DC", "DAYS_ALIVE_30",
}

print("Repository root:", ROOT)
print("Output directory:", OUT)

In [ ]:
def first_existing(paths):
    for path in paths:
        if Path(path).exists():
            return Path(path)
    return None


def find_col(df, candidates):
    lower = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower:
            return lower[cand.lower()]
    return None


def parse_first_columns_list(ipynb_path):
    if not Path(ipynb_path).exists():
        return []
    import nbformat
    nb = nbformat.read(ipynb_path, as_version=4)
    for cell in nb.cells:
        if cell.cell_type != "code":
            continue
        src = cell.source
        m = re.search(r"columns\s*=\s*(\[.*?\])", src, flags=re.S)
        if not m:
            continue
        try:
            val = ast.literal_eval(m.group(1))
        except Exception:
            continue
        if isinstance(val, list) and val:
            return [str(x) for x in val]
    return []


def normalize_treatment(series, dataset):
    s = series.copy()
    if dataset == "HYPERION":
        if s.dtype == object:
            txt = s.astype(str).str.strip().str.lower()
            return txt.isin(["1", "ttm", "hypothermia", "33", "33c", "yes", "true"]).astype(int)
        return pd.to_numeric(s, errors="coerce").fillna(0).astype(int)
    if s.dtype == object:
        txt = s.astype(str).str.strip().str.lower()
        return txt.isin(["1", "yes", "true", "t", "ttm", "hypothermia"]).astype(int)
    return (pd.to_numeric(s, errors="coerce").fillna(0) != 0).astype(int)


def format_missing(n_missing, denom):
    pct = 100 * n_missing / denom if denom else np.nan
    return f"{int(n_missing)}/{int(denom)} ({pct:.1f}%)" if denom else "0/0 (NA)"


def missingness_test(miss, treatment):
    tab = pd.crosstab(treatment, miss).reindex(index=[0, 1], columns=[False, True], fill_value=0)
    arr = tab.to_numpy()
    if arr.sum() == 0 or arr.shape != (2, 2):
        return np.nan, "not tested"
    try:
        _, p_chi, _, expected = chi2_contingency(arr, correction=False)
        if (expected < 5).any():
            _, p = fisher_exact(arr)
            return float(p), "Fisher exact"
        return float(p_chi), "chi-square"
    except Exception:
        return np.nan, "not tested"

## eTables 7-14: Power Simulation Tables

In [ ]:
power_input = first_existing(POWER_INPUT_CANDIDATES)
power_tables = {}
if power_input is None:
    print("No power_simulation_raw.csv found. Run summarized_results/powerSimulationAnalysis.ipynb first, then rerun this notebook.")
    power_raw = pd.DataFrame()
else:
    print("Loaded power input:", power_input)
    power_raw = pd.read_csv(power_input)
    display(power_raw.head())

def make_power_public_table(sub):
    out = sub.copy()
    rename = {
        "dataset": "Dataset",
        "outcome": "Outcome",
        "n": "N",
        "treat_prev": "TTM prevalence",
        "subgroup_prev": "Subgroup prevalence",
        "baseline_risk": "Baseline outcome risk",
        "abs_treatment_effect": "Absolute subgroup treatment effect",
        "n_sims": "Simulations",
        "n_successful_fits": "Successful fits",
        "n_failed_fits": "Failed fits",
        "power": "Power",
    }
    keep = [c for c in rename if c in out.columns]
    out = out[keep].rename(columns=rename)
    for col in ["TTM prevalence", "Subgroup prevalence", "Baseline outcome risk", "Absolute subgroup treatment effect", "Power"]:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce").round(3)
    for col in ["N", "Simulations", "Successful fits", "Failed fits"]:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce").astype("Int64")
    if "Failed fits" in out.columns and out["Failed fits"].fillna(0).eq(0).all():
        out = out.drop(columns=["Failed fits"])
    return out

if not power_raw.empty:
    order = [
        ("HYPERION", "mortality"),
        ("HYPERION", "neuro"),
        ("eICU", "mortality"),
        ("eICU", "neuro"),
        ("PMAP", "mortality"),
        ("PMAP", "neuro"),
        ("MIMIC-IV", "mortality"),
        ("MIMIC-IV", "neuro"),
    ]
    for i, (dataset, outcome) in enumerate(order, start=7):
        sub = power_raw[
            power_raw["dataset"].astype(str).eq(dataset)
            & power_raw["outcome"].astype(str).eq(outcome)
        ].copy()
        table = make_power_public_table(sub)
        name = f"eTable{i:02d}_power_{dataset.lower().replace('-', '').replace(' ', '_')}_{outcome}"
        power_tables[name] = table
        table.to_csv(OUT / f"{name}.csv", index=False)
        print(name, table.shape)
        display(table)

## eTables 16-19: Missingness by TTM Group

In [ ]:
missingness_public_tables = {}
missingness_audit_tables = {}

missing_order = [
    ("HYPERION", 16),
    ("eICU", 17),
    ("PMAP", 18),
    ("MIMIC-IV", 19),
]

for dataset, etable_num in missing_order:
    csv_path = first_existing(CSV[dataset])
    if csv_path is None:
        print(f"{dataset}: no predictor CSV found; skipping eTable {etable_num}.")
        continue
    df = pd.read_csv(csv_path)
    treat_col = find_col(df, TREATMENT_CANDIDATES[dataset])
    if treat_col is None:
        print(f"{dataset}: no treatment column found; skipping eTable {etable_num}.")
        continue
    treatment = normalize_treatment(df[treat_col], dataset)

    dml_cols = parse_first_columns_list(DML_NOTEBOOK[dataset])
    if dataset == "HYPERION" or not dml_cols:
        feature_cols = [
            c for c in df.columns
            if c not in OUTCOME_OR_ID_COLUMNS
            and c != treat_col
            and not c.lower().startswith("unnamed")
        ]
    else:
        feature_cols = [
            c for c in dml_cols
            if c in df.columns
            and c not in OUTCOME_OR_ID_COLUMNS
            and c != treat_col
            and not c.lower().startswith("unnamed")
        ]

    rows = []
    for col in feature_cols:
        miss = df[col].isna()
        no_ttm = treatment.eq(0)
        ttm = treatment.eq(1)
        p, test = missingness_test(miss, treatment)
        significant = bool(pd.notna(p) and p < 0.05)
        rows.append({
            "Variable": col,
            "No TTM missing": format_missing(miss[no_ttm].sum(), no_ttm.sum()),
            "TTM missing": format_missing(miss[ttm].sum(), ttm.sum()),
            "Difference": "*" if significant else "",
            "p_value": p,
            "test": test,
            "n_no_ttm": int(no_ttm.sum()),
            "n_ttm": int(ttm.sum()),
        })

    audit = pd.DataFrame(rows)
    public = audit[["Variable", "No TTM missing", "TTM missing", "Difference"]].copy()
    name = f"eTable{etable_num:02d}_missingness_{dataset.lower().replace('-', '').replace(' ', '_')}"
    public.to_csv(OUT / f"{name}.csv", index=False)
    audit.to_csv(OUT / f"{name}_audit_with_p_values.csv", index=False)
    missingness_public_tables[name] = public
    missingness_audit_tables[f"{name}_audit"] = audit
    print(name, public.shape, "source:", csv_path)
    display(public.head(40))

legend = pd.DataFrame({
    "Legend": [
        "Asterisk denotes a significant difference in the proportion of missing values between TTM and no-TTM groups (chi-square test, or Fisher's exact test when any expected cell count < 5; alpha = 0.05).",
        "The audit files include p_value and test columns for verification only; the public eTables omit those columns.",
    ]
})
legend.to_csv(OUT / "eTables16_19_missingness_legend.csv", index=False)
display(legend)

## Combined Workbook

In [ ]:
workbook = OUT / "supplement_etables_public.xlsx"
audit_workbook = OUT / "supplement_etables_audit.xlsx"

def safe_sheet(name):
    return re.sub(r"[^A-Za-z0-9_]", "_", name)[:31]

with pd.ExcelWriter(workbook) as writer:
    wrote = False
    for name, table in {**power_tables, **missingness_public_tables}.items():
        table.to_excel(writer, sheet_name=safe_sheet(name), index=False)
        wrote = True
    if not wrote:
        pd.DataFrame({"message": ["No tables were generated. Check input CSV availability."]}).to_excel(writer, sheet_name="README", index=False)

with pd.ExcelWriter(audit_workbook) as writer:
    wrote = False
    for name, table in missingness_audit_tables.items():
        table.to_excel(writer, sheet_name=safe_sheet(name), index=False)
        wrote = True
    if not wrote:
        pd.DataFrame({"message": ["No audit tables were generated. Check input CSV availability."]}).to_excel(writer, sheet_name="README", index=False)

print("Wrote", workbook)
print("Wrote", audit_workbook)
print("Output files:")
for path in sorted(OUT.glob("*")):
    print(" -", path.name)